# Star-$R$ jet graphs — visual feel

A few TopTagging **top** and **QCD** jets with the **star-$R$** hypergraph overlaid in $(\eta,\phi)$.

**Construction (training default $R=0.15$):** one hyperedge per active particle $c$, with support
$$
S_c = \{c\}\cup\{j:\Delta R_{cj}<R\}.
$$
Each hyperedge is drawn as **spokes** (line segments) from the center to its neighbors.
Particle marker **size** and **color** both encode energy (Blues colorbar).

Printed per jet: $N$ (particles), $E$ (star hyperedges), mean $|S|$, and spokes (= number of drawn center–neighbor segments).


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection

REPO = Path("..").resolve()
sys.path[:0] = [str(REPO), str(REPO / "src")]

from cpen.utils.graph_radius import (
    build_star_radius_hypergraph,
    eta_phi_from_four_vectors,
    particle_mask_from_four_vectors,
)
from cpen.utils.graph_radius_viz import (
    plot_star_center_hyperedge,
    reference_particle_index,
)
from cpen.utils.toptagging import (
    MAX_NUM_PARTICLES,
    N_PARTICLE_FEATURES,
    raw_hdf5_path,
)

import scienceplots  # noqa: F401

plt.style.use(["science", "no-latex"])
mpl.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 9,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
    }
)

FIG_DIR = Path("figures/star_graph_viz")
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/scratch/gpfs/BHANIN/jdezoort/datasets/toptagging")
SPLIT = "train"
NUM_PARTICLES = 128
RADIUS = 0.15
N_PER_CLASS = 3
SEED = 0

## Load a few jets

In [ ]:
hdf5_path = raw_hdf5_path(DATA_ROOT, SPLIT)
assert hdf5_path.is_file(), f"Missing {hdf5_path}"

table = pd.read_hdf(hdf5_path, key="table")
total_pf = MAX_NUM_PARTICLES * N_PARTICLE_FEATURES
labels = table.iloc[:, -1].to_numpy(dtype=np.int64)

rng = np.random.default_rng(SEED)
picks: dict[str, list[tuple[int, np.ndarray]]] = {"top": [], "QCD": []}
for class_label, name in [(1, "top"), (0, "QCD")]:
    pool = np.flatnonzero(labels == class_label)
    chosen = rng.choice(pool, size=N_PER_CLASS, replace=False)
    for idx in chosen:
        row = table.iloc[int(idx), :total_pf].to_numpy(dtype=np.float32)
        jet = row.reshape(MAX_NUM_PARTICLES, N_PARTICLE_FEATURES)[:NUM_PARTICLES]
        picks[name].append((int(idx), jet))

for name, items in picks.items():
    ns = [int(particle_mask_from_four_vectors(j).sum()) for _, j in items]
    print(f"{name}: idxs={[i for i, _ in items]}  N_active={ns}")

## Full star-$R$ overlay

One figure per jet. Gray spokes = all star hyperedges; particles colored/sized by energy.


In [ ]:
def star_graph_stats(jet: np.ndarray, *, radius: float) -> dict:
    build = build_star_radius_hypergraph(jet, radius=radius)
    active = build.active_indices
    n_e = int(build.incidence.shape[0])
    # Spokes = drawn center→neighbor segments (not the same as E hyperedges).
    n_spokes = 0
    for row in range(n_e):
        center = int(build.center_particles[row])
        members = np.flatnonzero(build.incidence[row] > 0)
        n_spokes += int(np.sum(members != center))
    mean_supp = float(build.particle_support_sizes.mean()) if n_e else 0.0
    return {
        "build": build,
        "active": active,
        "N": int(len(active)),
        "E": n_e,
        "mean_|S|": mean_supp,
        "spokes": n_spokes,
    }


def plot_star_r_overlay(
    jet: np.ndarray,
    *,
    radius: float,
    ax=None,
    title: str | None = None,
    size_scale: float = 280.0,
    edge_alpha: float = 0.22,
    edge_lw: float = 0.55,
):
    """Draw one jet; print N/E/mean|S|/spokes to stdout (not in the title)."""
    stats = star_graph_stats(jet, radius=radius)
    build = stats["build"]
    active = stats["active"]
    eta_all, phi_all = eta_phi_from_four_vectors(jet)
    eta, phi = eta_all[active], phi_all[active]
    energies = jet[active, 0].astype(np.float64)
    e_max = max(float(energies.max()), 1e-6)
    sizes = size_scale * energies / e_max

    g2a = {int(g): i for i, g in enumerate(active.tolist())}
    pos = np.stack([eta, phi], axis=1)

    segments = []
    for row in range(build.incidence.shape[0]):
        center = int(build.center_particles[row])
        if center not in g2a:
            continue
        cpos = pos[g2a[center]]
        members = np.flatnonzero(build.incidence[row] > 0)
        for m in members:
            m = int(m)
            if m == center or m not in g2a:
                continue
            segments.append((cpos, pos[g2a[m]]))

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(figsize=(4.2, 3.8), dpi=800)
    else:
        fig = ax.figure

    if segments:
        ax.add_collection(
            LineCollection(
                segments,
                colors="#666666",
                linewidths=edge_lw,
                alpha=edge_alpha,
                zorder=2,
            )
        )

    sc = ax.scatter(
        eta,
        phi,
        s=sizes,
        c=energies,
        cmap="Blues",
        alpha=0.9,
        edgecolors="k",
        linewidths=0.25,
        zorder=3,
        vmin=0.0,
        vmax=e_max,
    )
    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(r"energy $E$")

    ax.set_xlabel(r"$\eta$")
    ax.set_ylabel(r"$\phi$")
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(True, which="both", ls=":", alpha=0.35)
    if title is not None:
        ax.set_title(title)

    print(
        f"{title or 'jet'}: "
        f"N={stats['N']}, E={stats['E']} (star hyperedges), "
        f"mean |S|={stats['mean_|S|']:.2f}, "
        f"spokes={stats['spokes']} (center–neighbor segments)"
    )
    return build, fig, ax


In [ ]:
# One figure per example jet (saved separately).
for name in ["top", "QCD"]:
    for idx, jet in picks[name]:
        build, fig, ax = plot_star_r_overlay(
            jet,
            radius=RADIUS,
            title=rf"{name} jet {idx}",
        )
        fig.tight_layout()
        outfile = FIG_DIR / f"star_r{RADIUS:g}_{name}_{idx}"
        fig.savefig(outfile.with_suffix(".pdf"), bbox_inches="tight")
        fig.savefig(outfile.with_suffix(".png"), bbox_inches="tight")
        plt.show()
        print("saved", outfile.with_suffix(".pdf"))


## One star in focus

Same jets, but only the hyperedge **centered on the highest-$E$ particle** (thicker spokes).
Still one figure per jet; particles colored/sized by energy (no star marker).


In [ ]:
def plot_maxE_star_focus(
    jet: np.ndarray,
    *,
    radius: float,
    title: str | None = None,
    size_scale: float = 280.0,
):
    build = build_star_radius_hypergraph(jet, radius=radius)
    ref = reference_particle_index(jet)
    active = build.active_indices
    eta_all, phi_all = eta_phi_from_four_vectors(jet)
    eta, phi = eta_all[active], phi_all[active]
    energies = jet[active, 0].astype(np.float64)
    e_max = max(float(energies.max()), 1e-6)
    sizes = size_scale * energies / e_max
    g2a = {int(g): i for i, g in enumerate(active.tolist())}

    fig, ax = plt.subplots(figsize=(4.2, 3.8))

    # All particles (muted) + focus star spokes via helper underlay then recolor.
    ax.scatter(
        eta,
        phi,
        s=sizes * 0.55,
        c="0.85",
        alpha=0.5,
        edgecolors="none",
        zorder=1,
        linewidths=0,
    )

    # Draw only the max-E centered hyperedge spokes.
    row = int(np.flatnonzero(build.center_particles == ref)[0]) if ref in set(build.center_particles.tolist()) else None
    n_spokes = 0
    if row is not None and ref in g2a:
        cpos = np.array([eta[g2a[ref]], phi[g2a[ref]]])
        members = np.flatnonzero(build.incidence[row] > 0)
        segments = []
        for m in members:
            m = int(m)
            if m == ref or m not in g2a:
                continue
            segments.append((cpos, np.array([eta[g2a[m]], phi[g2a[m]]])))
        n_spokes = len(segments)
        if segments:
            ax.add_collection(
                LineCollection(
                    segments,
                    colors="#444444",
                    linewidths=1.2,
                    alpha=0.75,
                    zorder=2,
                )
            )

    sc = ax.scatter(
        eta,
        phi,
        s=sizes,
        c=energies,
        cmap="Blues",
        alpha=0.9,
        edgecolors="k",
        linewidths=0.25,
        zorder=3,
        vmin=0.0,
        vmax=e_max,
    )
    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(r"energy $E$")

    mean_supp = float(build.particle_support_sizes.mean()) if build.incidence.shape[0] else 0.0
    ax.set_xlabel(r"$\eta$")
    ax.set_ylabel(r"$\phi$")
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(True, which="both", ls=":", alpha=0.35)
    if title is not None:
        ax.set_title(title)

    print(
        f"{title or 'jet'}: "
        f"N={len(active)}, E={int(build.incidence.shape[0])} (star hyperedges), "
        f"mean |S|={mean_supp:.2f}, "
        f"spokes={n_spokes} (max-$E$ center–neighbor segments only)"
    )
    return fig, ax


for name in ["top", "QCD"]:
    for idx, jet in picks[name]:
        fig, ax = plot_maxE_star_focus(
            jet,
            radius=RADIUS,
            title=rf"{name} jet {idx} — max-$E$ star",
        )
        fig.tight_layout()
        outfile = FIG_DIR / f"star_r{RADIUS:g}_{name}_{idx}_maxE"
        fig.savefig(outfile.with_suffix(".pdf"), bbox_inches="tight")
        fig.savefig(outfile.with_suffix(".png"), bbox_inches="tight")
        plt.show()
        print("saved", outfile.with_suffix(".pdf"))
